<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 12 · Validation and Failure Modes

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/12_dxquant_validation_and_failure_modes.ipynb)


## Notebook Goals

DXQuant is designed to fail loudly when inputs are inconsistent or numerically dangerous. This
notebook collects a set of small examples showing what kinds of issues are caught early.

All examples catch exceptions and print short messages so the notebook runs end-to-end.

## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## 1. Invalid Process Parameters

Processes validate obvious misuse early (for example negative volatility).

In [ ]:
try:
    _ = dx.GeometricBrownianMotion(drift=0.02, volatility=-0.1)
except Exception as exc:
    print("GBM init error:", exc)

## 2. Invalid Time Grids

Simulation grids must be strictly increasing and start at 0.

In [ ]:
gbm = dx.GeometricBrownianMotion(drift=0.02, volatility=0.2, seed=1)

try:
    bad_grid = np.array([0.1, 0.2, 0.3])
    _ = gbm.simulate_paths(spot=100.0, time_grid=bad_grid, paths=1000)
except Exception as exc:
    print("grid error:", exc)

try:
    bad_grid = np.array([0.0, 0.5, 0.5, 1.0])
    _ = gbm.simulate_paths(spot=100.0, time_grid=bad_grid, paths=1000)
except Exception as exc:
    print("grid error:", exc)

## 3. Curve Construction Errors

Curves validate basic requirements such as node counts and consistent dates.

In [ ]:
pricing_date = dt.date(2024, 1, 1)

try:
    _ = dx.InterpolatedZeroCurve(
        name="bad",
        reference_date=pricing_date,
        nodes=[(dt.date(2025, 1, 1), 0.02)],
    )
except Exception as exc:
    print("zero curve error:", exc)

try:
    _ = dx.ConstantShortRateCurve("disc", pricing_date, rate=-0.01)
except Exception as exc:
    print("short rate error:", exc)

## 4. Interpolator Validation

`Spline1D` checks basic constraints such as monotonic x-values and minimum node counts.

In [ ]:
try:
    x = np.array([0.0, 1.0, 1.0, 2.0])
    y = np.array([0.02, 0.03, 0.03, 0.04])
    _ = dx.Spline1D(x, y, degree=3)
except Exception as exc:
    print("spline error:", exc)

## 5. Portfolio Aggregation Preconditions

Portfolio aggregation assumes comparable Monte Carlo path counts across positions. If the
path counts differ, aggregation is rejected.

In [ ]:
env = dx.MarketEnvironment(name="val", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, 0.02))

pricer_a = dx.MonteCarloPricer(
    env=env,
    process=dx.GeometricBrownianMotion(drift=0.02, volatility=0.2, seed=1),
    discount_curve_key="discount",
    paths=5000,
    steps=16,
)

pricer_b = dx.MonteCarloPricer(
    env=env,
    process=dx.GeometricBrownianMotion(drift=0.02, volatility=0.2, seed=2),
    discount_curve_key="discount",
    paths=6000,
    steps=16,
)

payoff = dx.EuropeanCallPayoff(strike=105.0)
maturity = dt.date(2025, 1, 1)

res_a = pricer_a.price_terminal(payoff, maturity=maturity, spot=100.0)
res_b = pricer_b.price_terminal(payoff, maturity=maturity, spot=100.0)

try:
    _ = res_a.discounted_payoffs + res_b.discounted_payoffs
except Exception as exc:
    print("array shape mismatch:", exc)

print(
    "paths a/b:", res_a.discounted_payoffs.size, res_b.discounted_payoffs.size
)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">